# 🇵🇭 Philippine Address Parsing & Normalization Pipeline

**Architecture**

```
Raw addresses
     │
     ▼
Stage 1 ── Alias normalization      (ph_address_alias_extended_v4.csv)
     │
     ▼
Stage 2 ── PSGC fuzzy match          (rapidfuzz · province-anchored)
     │
     ▼
Stage 3 ── Confidence gate           (high → fast path · low → API)
     │                    │
     ▼                    ▼
  dim_location        Stage 4 ── Nominatim OSM API verify
  fast lookup              │
     │                    ▼
     └──────── Stage 5 ──────── Output split
                               │               │
                     verified_addresses.xlsx   invalid_addresses.xlsx
```

**Key design rules**
- A street/barangay token that *happens* to share a city name is **not** assigned that city unless a province or region corroborates it (fixes the *South Signal → General Santos* false match)
- Only ambiguous / low-confidence rows hit the Nominatim API — high-confidence rows are resolved purely from `dim_location` for speed
- 10 000 rows target: < 20 min total


## Cell 0 — Environment check
Verify all required packages are present before running the pipeline.

In [31]:
import importlib, sys

REQUIRED = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'rapidfuzz': 'rapidfuzz',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
}

missing = []
for pkg_label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  ✅  {pkg_label:<14} {ver}')
    except ImportError:
        print(f'  ❌  {pkg_label:<14} NOT FOUND')
        missing.append(pkg)

if missing:
    print(f'\n⚠️  Install missing packages:')
    print(f'    pip install {" ".join(missing)}')
else:
    print('\n✅  All dependencies satisfied — ready to run.')


  ✅  pandas         3.0.1
  ✅  numpy          2.4.3
  ✅  rapidfuzz      3.14.3
  ✅  tqdm           4.67.3
  ✅  openpyxl       3.1.5
  ✅  xlsxwriter     3.2.9

✅  All dependencies satisfied — ready to run.


## Cell 1 — Configuration

Edit the paths and thresholds here. Everything else in the pipeline reads from these variables.

| Variable | Default | Meaning |
|---|---|---|
| `CITY_SCORE_HIGH` | 88 | Auto-accept without API call |
| `CITY_SCORE_LOW` | 65 | Minimum to even consider a city match |
| `PROV_SCORE_HIGH` | 88 | Auto-accept province |
| `PROV_SCORE_LOW` | 60 | Minimum province score |
| `USE_API` | `True` | Set `False` for fuzzy-only mode (faster, less accurate) |
| `API_QUERY_MODE` | `'aggressive'` | API search mode: `'strict'` = minimal queries, `'aggressive'` = typo-tolerant fallbacks |
| `MAX_ROWS` | `None` | Set an integer (e.g. `100`) to process only a slice for testing |


In [32]:
from pathlib import Path

# -- File paths --
BASE_DIR = Path("..") / ".." / "data"   # notebook is in update_address/notebooks

INPUT_FILE   = str(BASE_DIR / "sample"  / "sample_org_address.xlsx")
DIM_LOCATION = str(BASE_DIR / "mapping" / "dim_location_20260316_v2.csv")
ALIAS_FILE   = str(BASE_DIR / "utils"   / "ph_address_alias_extended_v4.csv")
OUT_VERIFIED = str(BASE_DIR / "output"  / "verified_addresses.xlsx")
OUT_INVALID  = str(BASE_DIR / "output"  / "invalid_addresses.xlsx")

# -- Batch input (unchanged) --------------------------------------------------
input_paths = [
    Path('../../data/sample/Structured_Philippine_Addresses_Unmatched/') / f'Structured_Philippine_Addresses_Unmatched_part_{i:03d}.xlsx'
    for i in range(1, 11)   # Adjust range to cover your batches, e.g. range(1, 101)
]

# -- Fuzzy-match thresholds (0-100) ------------------------------------------
CITY_SCORE_HIGH  = 88
CITY_SCORE_LOW   = 65
PROV_SCORE_HIGH  = 88
PROV_SCORE_LOW   = 60
BRGY_SCORE_MIN   = 75

# -- API settings kept for optional future use (currently disabled) -----------
PHOTON_URL = 'https://photon.komoot.io/api'
PHOTON_UA  = 'ph-address-pipeline/1.0 (research)'
BATCH_DELAY = 1.1
MAX_RETRIES = 1

# -- Run controls -------------------------------------------------------------
USE_API        = False         # Force fuzzy-only mode for maximum speed
API_QUERY_MODE = 'strict'      # Placeholder when API is re-enabled
MAX_ROWS       = None          # None = all rows; set e.g. 100 for quick test runs

# -- Quick path check ---------------------------------------------------------
if input_paths:
    print(f'  Batch mode enabled: {len(input_paths)} input files')
    for p in input_paths:
        status = 'OK' if p.exists() else 'NOT FOUND'
        print(f'  {status}  {p}')
else:
    status = 'OK' if Path(INPUT_FILE).exists() else 'NOT FOUND'
    print(f'  {status}  {INPUT_FILE}')

for f in [DIM_LOCATION, ALIAS_FILE]:
    status = 'OK' if Path(f).exists() else 'NOT FOUND'
    print(f'  {status}  {f}')
print(f'  USE_API = {USE_API} (fuzzy-only)')


  Batch mode enabled: 10 input files
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_001.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_002.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_003.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_004.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_005.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_006.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_part_007.xlsx
  OK  ..\..\data\sample\Structured_Philippine_Addresses_Unmatched\Structured_Philippine_Addresses_Unmatched_p

## Cell 2 — Imports & logging setup

In [33]:
import re
import time
import logging
import urllib.request
import urllib.parse
import json
from typing import Optional

import pandas as pd
import numpy as np
from IPython.display import display
from rapidfuzz import fuzz, process
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_pipeline')
log.info('Imports OK')


15:20:28  INFO      Imports OK


## Cell 3 — Load reference data

Loads `dim_location` (42 000+ PSGC barangay rows) and `ph_address_alias` (499 alias pairs), then builds fast in-memory lookup lists for fuzzy matching.

In [34]:
def _read_csv_with_fallback(path: str) -> pd.DataFrame:
    strict_encodings = ['utf-8', 'utf-8-sig']
    for enc in strict_encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    # Keep mostly-UTF8 files readable even if they contain a few broken bytes.
    try:
        log.warning(f'Loaded {path} using utf-8 with replacement for invalid bytes')
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    except Exception:
        pass

    for enc in ['cp1252', 'latin1']:
        try:
            log.warning(f'Loaded {path} using fallback encoding: {enc}')
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f'Unable to decode CSV file: {path}')


def load_reference(dim_path: str, alias_path: str):
    log.info('Loading dim_location ...')
    dim = _read_csv_with_fallback(dim_path)
    str_cols = ['region_name', 'province_name', 'city_name', 'barangay_name']
    for c in str_cols:
        dim[c] = dim[c].fillna('').astype(str).str.upper().str.strip()

    log.info('Loading alias table ...')
    alias = _read_csv_with_fallback(alias_path)
    alias['pattern'] = alias['pattern'].fillna('').astype(str).str.upper().str.strip()
    alias['replacement'] = alias['replacement'].fillna('').astype(str).str.upper().str.strip()

    cities = sorted(x for x in dim['city_name'].unique().tolist() if x)
    provinces = sorted(x for x in dim['province_name'].unique().tolist() if x)
    regions = sorted(x for x in dim['region_name'].unique().tolist() if x)
    barangays = sorted(x for x in dim['barangay_name'].unique().tolist() if x)

    log.info(
        f'dim_location: {len(dim):,} rows | '
        f'{len(cities)} cities | {len(provinces)} provinces | '
        f'{len(regions)} regions | {len(barangays):,} barangays'
    )
    return dim, alias, cities, provinces, regions, barangays


dim, alias_df, cities, provinces, regions, barangays = load_reference(
    DIM_LOCATION, ALIAS_FILE
)

print('\n-- Sample dim_location --')
display(dim.head(3))
print('\n-- Sample aliases --')
display(alias_df.head(10))


15:20:28  INFO      Loading dim_location ...
15:20:29  WARNING   Loaded ..\..\data\mapping\dim_location_20260316_v2.csv using utf-8 with replacement for invalid bytes
15:20:29  INFO      Loading alias table ...
15:20:29  WARNING   Loaded ..\..\data\utils\ph_address_alias_extended_v4.csv using utf-8 with replacement for invalid bytes
15:20:29  INFO      dim_location: 42,011 rows | 1426 cities | 83 provinces | 18 regions | 25,843 barangays



-- Sample dim_location --


,psgc_10_digit,region_name,province_name,city_name,barangay_name,region_code,province_code,city_code,barangay_code
0,1400101001,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,AGTANGAO,14,1,1,1
1,1400101002,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,ANGAD,14,1,1,2
2,1400101003,CORDILLERA ADMINISTRATIVE REGION (CAR),ABRA,BANGUED,BAÑACAO,14,1,1,3



-- Sample aliases --


,pattern,replacement
0,BRGY,BARANGAY
1,BRGY.,BARANGAY
2,BRG,BARANGAY
3,BRG.,BARANGAY
4,BARRIO,BARANGAY
5,BARRIO.,BARANGAY
6,POB,POBLACION
7,POB.,POBLACION
8,POBL,POBLACION
9,POBL.,POBLACION


## Cell 4 — Stage 1: Alias normalization

Builds a **single-pass compiled regex** from all 499 alias pairs (sorted longest-first so multi-word patterns like `CITY OF MARIKINA` fire before the single-token `MARIKINA`).

A post-pass deduplication step catches chained-alias artifacts like `CITY CITY` or `BARANGAY BARANGAY` that arise when both `CITY OF X → X CITY` and `X → X CITY` fire on the same string.

In [35]:
def build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (p.strip(), r.strip())
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if isinstance(p, str) and p.strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest pattern first
    replace_map = {p: r for p, r in pairs}
    pattern  = '|'.join(re.escape(p) for p, _ in pairs)
    compiled = re.compile(r'\b(' + pattern + r')\b')
    return compiled, replace_map


def normalize_alias(text: str, compiled_re, replace_map: dict) -> str:
    text = str(text).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    text = compiled_re.sub(lambda m: replace_map.get(m.group(0), m.group(0)), text)
    # Remove duplicate consecutive words (e.g. 'CITY CITY', 'BARANGAY BARANGAY')
    text = re.sub(
        r'\b(CITY|BARANGAY|STREET|AVENUE|BOULEVARD|VILLAGE|PROVINCE)\s+\1\b',
        r'\1', text, flags=re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', text).strip()


compiled_re, replace_map = build_alias_regex(alias_df)
log.info(f'Alias regex built from {len(alias_df)} patterns')

# ── Quick test on sample addresses ───────────────────────────────────────────
test_cases = [
    'sapang palay sjdm',
    '112 Ballecer st Zone South Signal Village Taguig',
    '55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City',
    'loilo City',
    'carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL',
]
print(f'{"ORIGINAL":<55}  {"NORMALIZED"}')
print('─' * 110)
for t in test_cases:
    print(f'{t:<55}  {normalize_alias(t, compiled_re, replace_map)}')


15:20:29  INFO      Alias regex built from 504 patterns


ORIGINAL                                                 NORMALIZED
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
sapang palay sjdm                                        SAPANG PALAY SAN JOSE DEL MONTE CITY
112 Ballecer st Zone South Signal Village Taguig         112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
55 Dawn St South Kim View Park SSS Village Concepcionh 2 MArikina City  55 DAWN STREET SOUTH KIM VIEW PARK SSS VILLAGE CONCEPCIONH 2 MARIKINA CITY
loilo City                                               LOILO CITY
carmencdoc Carmen CAGAYAN DE ORO CITY (Capital) MISAMIS ORIENTAL  CARMENCDOC CARMEN CAGAYAN DE ORO CITY (CAPITAL) MISAMIS ORIENTAL


## Cell 5 — Stage 2: Fuzzy matching helpers

Two functions used throughout the pipeline:
- **`best_match`** — matches a single query string against a list (full-string scorer)
- **`token_scan`** — splits the address into tokens and tries each token independently; returns the highest-scoring hit across all tokens

Both use `rapidfuzz.fuzz.WRatio` which handles partial overlaps well (e.g. `QUEZON CITY` inside a longer address string).

In [36]:
def best_match(
    query: str,
    choices: list,
    scorer=fuzz.WRatio,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Return (best_match_string, score) or (None, 0) if below cutoff."""
    if not query:
        return None, 0
    result = process.extractOne(query, choices, scorer=scorer,
                                score_cutoff=score_cutoff)
    return (result[0], int(result[1])) if result else (None, 0)


def token_scan(
    tokens: list,
    choices: list,
    score_cutoff: int = 0,
) -> tuple[Optional[str], int]:
    """Try each token against choices; return best (match, score)."""
    best_s, best_m = 0, None
    for tok in tokens:
        if len(tok) < 3:
            continue
        m, s = best_match(tok, choices, score_cutoff=score_cutoff)
        if s > best_s:
            best_s, best_m = s, m
    return best_m, best_s


# ── Quick smoke test ──────────────────────────────────────────────────────────
tests = [
    ('QUEZON CITY', cities),
    ('ILOCOS SUR', provinces),
    ('TAGUIG', cities),
    ('SOUTH SIGNAL', cities),   # should score low / return nothing meaningful
]
print(f'{"Query":<25}  {"Best match":<35}  Score')
print('─' * 75)
for q, lst in tests:
    m, s = best_match(q, lst, score_cutoff=0)
    print(f'{q:<25}  {str(m):<35}  {s}')


Query                      Best match                           Score
───────────────────────────────────────────────────────────────────────────
QUEZON CITY                QUEZON CITY                          100
ILOCOS SUR                 ILOCOS SUR                           100
TAGUIG                     CITY OF TAGUIG                       90
SOUTH SIGNAL               SIGAY                                72


## Cell 6 — Stage 2 + 3: Core address matching & confidence gate

### Strict city-detection rule
A token scoring ≥ `CITY_SCORE_LOW` is a **candidate**, not a result.  
If that candidate city exists in multiple provinces (ambiguous), it **requires** the province to also appear in the address with score ≥ `PROV_SCORE_LOW` before the city is assigned.  
This is what prevents `South Signal Village` from being mapped to `City of General Santos` — `SIGNAL` has no city match, and the correct `TAGUIG` token maps cleanly.

### Confidence levels
| Level | Condition | Action |
|---|---|---|
| `high` | city score ≥ 88 | Accept immediately — no API |
| `medium` | city score 65–87 | Send to Nominatim for verification |
| `low` | no city, province score < 88 | Send to Nominatim |
| `none` | nothing matched | Mark invalid |


In [37]:
def match_address(
    address_norm: str,
    dim: pd.DataFrame,
    cities: list, provinces: list, regions: list, barangays: list,
    city_high: int = CITY_SCORE_HIGH,
    city_low: int  = CITY_SCORE_LOW,
    prov_high: int = PROV_SCORE_HIGH,
    prov_low: int  = PROV_SCORE_LOW,
) -> dict:
    result = dict(
        city_name=None, city_code=None, city_score=0,
        province_name=None, province_code=None, province_score=0,
        region_name=None, region_code=None,
        barangay_name=None, barangay_code=None,
        psgc_10_digit=None,
        confidence='none', needs_api=False,
    )

    tokens = [t for t in re.split(r'[,\s]+', address_norm) if len(t) >= 3]

    # ── Build context hints first (used for robust barangay disambiguation) ─────
    prov_hint, prov_hint_score = token_scan(tokens, provinces, score_cutoff=prov_low)
    city_full_hint, city_full_score = best_match(address_norm, cities, score_cutoff=city_low)
    city_tok_hint, city_tok_score = token_scan(tokens, cities, score_cutoff=city_low)
    city_hint, city_hint_score = (
        (city_full_hint, city_full_score)
        if city_full_score >= city_tok_score
        else (city_tok_hint, city_tok_score)
    )

    def _name_overlaps_address(name: str, text: str) -> bool:
        skip = {'CITY', 'MUNICIPALITY', 'OF', 'PROVINCE'}
        words = [w for w in str(name).split() if w not in skip and len(w) >= 4]
        return any(w in text for w in words)

    generic_brgy_terms = {
        'PUROK', 'POBLACION', 'ZONE', 'CENTRO', 'PROPER',
        'BARANGAY', 'SITIO', 'VILLA', 'DISTRICT', 'AREA'
    }

    def _best_barangay_row(brgy_name: Optional[str], brgy_raw_score: int, source: str):
        if not brgy_name:
            return None, -1, -1
        subset = dim[dim['barangay_name'] == brgy_name]
        if subset.empty:
            return None, -1, -1

        best_row = None
        best_adjusted = -1
        best_bonus = -1
        for _, row in subset.iterrows():
            bonus = 0
            if city_hint and row['city_name'] == city_hint:
                bonus += 25
            if prov_hint and row['province_name'] == prov_hint:
                bonus += 20
            if _name_overlaps_address(row['city_name'], address_norm):
                bonus += 10
            if _name_overlaps_address(row['province_name'], address_norm):
                bonus += 8

            adjusted = brgy_raw_score + bonus
            if adjusted > best_adjusted:
                best_adjusted = adjusted
                best_bonus = bonus
                best_row = row

        # Accept barangay anchor only when context supports it.
        # Prevents generic-token false positives like PUROK/BARANG/LUZON/etc.
        brgy_upper = str(brgy_name).upper().strip()
        brgy_words = [w for w in brgy_upper.split() if w]
        is_generic = brgy_upper in generic_brgy_terms
        is_short_single = len(brgy_words) == 1 and len(brgy_upper) < 7
        looks_placeholder = brgy_upper.startswith('BARANG')

        if source == 'token' and best_bonus < 10:
            return None, -1, -1
        if is_generic and best_bonus < 15:
            return None, -1, -1
        if (is_short_single or looks_placeholder) and best_bonus < 20:
            return None, -1, -1
        if len(brgy_words) == 1 and source == 'full' and best_bonus < 10:
            return None, -1, -1

        return best_row, best_adjusted, best_bonus

    # ── PRIORITY 1: Barangay-first with context-aware candidate scoring ─────────
    # Generic behavior: handles all addresses where a misleading token competes
    # with a true multi-word barangay (e.g., LUZON vs BATASAN HILLS).
    brgy_full, brgy_score_full = best_match(address_norm, barangays, score_cutoff=BRGY_SCORE_MIN)
    brgy_tok,  brgy_score_tok  = token_scan(tokens, barangays, score_cutoff=BRGY_SCORE_MIN)

    row_full, adj_full, bonus_full = _best_barangay_row(brgy_full, brgy_score_full, source='full')
    row_tok,  adj_tok,  bonus_tok  = _best_barangay_row(brgy_tok, brgy_score_tok, source='token')

    if max(adj_full, adj_tok) >= BRGY_SCORE_MIN:
        brgy_row = row_full if adj_full >= adj_tok else row_tok
        result.update(
            city_name=brgy_row['city_name'],
            city_code=int(brgy_row['city_code']),
            city_score=95,
            province_name=brgy_row['province_name'],
            province_code=int(brgy_row['province_code']),
            province_score=95,
            region_name=brgy_row['region_name'],
            region_code=int(brgy_row['region_code']),
            barangay_name=brgy_row['barangay_name'],
            barangay_code=int(brgy_row['barangay_code']),
            psgc_10_digit=int(brgy_row['psgc_10_digit']),
            confidence='high', needs_api=False,
        )
        return result

    # ── PRIORITY 2: Fall back to province + city matching if no barangay ──────
    prov_match, prov_score = prov_hint, prov_hint_score

    # ── City scan (full string + token scan; take best) ───────────────────
    city_match, city_score = city_hint, city_hint_score

    # ── Strict disambiguation ─────────────────────────────────────────────
    # Always require province corroboration for cities that exist in >1 province.
    if city_match:
        candidate_provs = dim[dim['city_name'] == city_match]['province_name'].unique()
        if len(candidate_provs) > 1:
            if not (prov_match and prov_match in candidate_provs
                    and prov_score >= prov_low):
                city_score = 0
                city_match = None

    # ── Populate result from dim_location ────────────────────────────────
    if city_match:
        subset = dim[dim['city_name'] == city_match]
        if prov_match and prov_score >= prov_low:
            sub2 = subset[subset['province_name'] == prov_match]
            if not sub2.empty:
                subset = sub2
        first = subset.iloc[0]
        result.update(
            city_name=city_match, city_code=int(first['city_code']),
            city_score=city_score,
            province_name=first['province_name'], province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )
    elif prov_match and prov_score >= prov_high:
        subset = dim[dim['province_name'] == prov_match]
        first = subset.iloc[0]
        result.update(
            province_name=prov_match, province_code=int(first['province_code']),
            province_score=prov_score,
            region_name=first['region_name'], region_code=int(first['region_code']),
        )

    # ── Confidence classification ─────────────────────────────────────────
    cs = result['city_score']
    ps = result['province_score']
    if cs >= city_high:
        result['confidence'], result['needs_api'] = 'high', False
    elif cs >= city_low:
        result['confidence'], result['needs_api'] = 'medium', True
    elif result['province_name'] and ps >= prov_high:
        result['confidence'], result['needs_api'] = 'medium', True
    else:
        result['confidence'], result['needs_api'] = 'low', True

    return result


# ── Generalized regression tests (not single-case tuning) ────────────────────
test_addresses = [
    '19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY',
    '112 Ballecer st Zone South Signal Village Taguig',
]

for idx, raw in enumerate(test_addresses, 1):
    addr = normalize_alias(raw, compiled_re, replace_map)
    ans = match_address(addr, dim, cities, provinces, regions, barangays)
    print(f'Test {idx}')
    print(f'  Input      : {raw}')
    print(f'  Normalized : {addr}')
    print(f'  Barangay   : {ans["barangay_name"]}')
    print(f'  City       : {ans["city_name"]}')
    print(f'  Province   : {ans["province_name"]}')
    print(f'  Confidence : {ans["confidence"]}')
    print('-' * 80)


Test 1
  Input      : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Normalized : 19 LUZON STREET BATASAN HILLS 67C JACSON STREET QUEZON CITY
  Barangay   : BATASAN HILLS
  City       : QUEZON CITY
  Province   : METRO MANILA
  Confidence : high
--------------------------------------------------------------------------------
Test 2
  Input      : 112 Ballecer st Zone South Signal Village Taguig
  Normalized : 112 BALLECER STREET ZONE SOUTH SIGNAL VILLAGE TAGUIG CITY
  Barangay   : SOUTH SIGNAL VILLAGE
  City       : CITY OF TAGUIG
  Province   : METRO MANILA
  Confidence : high
--------------------------------------------------------------------------------


## Cell 7 — Run Stage 1–3 on all input rows

Loads the input file, applies alias normalization, then runs the fuzzy matcher on every row. Results are split into `high_conf` (no API needed) and `low_conf` (needs Nominatim verification).

In [38]:
import time

t_start = time.time()

if input_paths:
    existing_paths = [p for p in input_paths if p.exists()]
    missing_paths = [p for p in input_paths if not p.exists()]

    for mp in missing_paths:
        log.warning(f'Missing batch file: {mp}')

    if not existing_paths:
        raise FileNotFoundError('No batch files found in input_paths.')

    log.info(f'Loading batch files: {len(existing_paths)} found')
    frames = []
    for p in existing_paths:
        tmp = pd.read_excel(p)
        tmp['_batch_file'] = p.name
        frames.append(tmp)
    df_input = pd.concat(frames, ignore_index=True)
else:
    log.info(f'Loading {INPUT_FILE} ...')
    df_input = pd.read_excel(INPUT_FILE)

if MAX_ROWS:
    df_input = df_input.iloc[:MAX_ROWS]
    log.info(f'Limiting to {MAX_ROWS} rows (MAX_ROWS is set)')

addr_col = df_input.columns[0]
log.info(f'Address column: "{addr_col}"  |  Total rows: {len(df_input):,}')

# -- Stage 1: alias normalization ---------------------------------------------
log.info('Stage 1: alias normalization ...')
addresses = df_input[addr_col].fillna('').astype(str).tolist()
normalized = [normalize_alias(x, compiled_re, replace_map) for x in addresses]
df_input['address_normalized'] = normalized

# -- Stage 2-3: fuzzy match + confidence gate ---------------------------------
log.info('Stage 2-3: fuzzy matching + confidence gate ...')
records = []
batch_files = df_input['_batch_file'].tolist() if '_batch_file' in df_input.columns else None

for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy match', unit='row')):
    rec = {
        'original_address': addresses[i],
        'address_normalized': addr_norm,
        'api_status': 'skipped',
        'osm_city': None, 'osm_province': None, 'osm_region': None,
        'osm_display': None, 'osm_lat': None, 'osm_lon': None,
    }
    if batch_files is not None:
        rec['batch_file'] = batch_files[i]
    rec.update(match_address(
        addr_norm,
        dim, cities, provinces, regions, barangays,
    ))
    rec['needs_api'] = False
    records.append(rec)

high_conf = [r for r in records if r.get('confidence') == 'high']
low_conf = [r for r in records if r.get('confidence') != 'high']

t_fuzzy = time.time() - t_start
log.info(f'Fuzzy pass done in {t_fuzzy:.1f}s')
log.info(f'  High confidence : {len(high_conf):,}')
log.info(f'  Lower confidence: {len(low_conf):,}')
log.info('  API stage is disabled; fuzzy-only output will be used')

# -- Preview ------------------------------------------------------------------
preview_cols = [
    'original_address', 'address_normalized',
    'city_name', 'province_name', 'confidence', 'city_score'
]
if records and 'batch_file' in records[0]:
    preview_cols = ['batch_file'] + preview_cols

df_preview = pd.DataFrame(records)[preview_cols]
display(df_preview)


15:20:29  INFO      Loading batch files: 10 found
15:20:30  INFO      Address column: "Original_Address"  |  Total rows: 10,000
15:20:30  INFO      Stage 1: alias normalization ...
15:20:30  INFO      Stage 2-3: fuzzy matching + confidence gate ...


Fuzzy match:   0%|          | 0/10000 [00:00<?, ?row/s]

15:45:40  INFO      Fuzzy pass done in 1510.5s
15:45:40  INFO        High confidence : 5,673
15:45:40  INFO        Lower confidence: 4,327
15:45:40  INFO        API stage is disabled; fuzzy-only output will be used


,batch_file,original_address,address_normalized,city_name,province_name,confidence,city_score
0,Structured_Philippine_Addresses_Unmatched_part...,LIONS PARK RES SUNVALLEY PQUE,LIONS PARK RES SUNVALLEY PARA�AQUE CITY,CITY OF PARA�AQUE,METRO MANILA,high,95
1,Structured_Philippine_Addresses_Unmatched_part...,"Villa Del Rio Babag 2, Blk 10, Lot 35","VILLA DEL RIO BABAG 2, BLOCK 10, LOT 35",CITY OF BUTUAN,AGUSAN DEL NORTE,high,95
2,Structured_Philippine_Addresses_Unmatched_part...,zone 1 upper balulng,ZONE 1 UPPER BALULNG,CITY OF DIGOS,DAVAO DEL SUR,high,95
3,Structured_Philippine_Addresses_Unmatched_part...,"House 4 Adolfo Compound,Highway 11 St Brgy Tal...","HOUSE 4 ADOLFO COMPOUND,HIGHWAY 11 STREET BARA...",BALAMBAN,CEBU,medium,87
4,Structured_Philippine_Addresses_Unmatched_part...,Quest Hotel,QUEST HOTEL,CITY OF OROQUIETA,MISAMIS OCCIDENTAL,medium,72
...,...,...,...,...,...,...,...
9995,Structured_Philippine_Addresses_Unmatched_part...,"BLK. 27 LOT 7,9,11 VERBENA ST. CAMILLA HOMES","BLOCK. 27 LOT 7,9,11 VERBENA STREET. CAMILLA H...",CITY OF PARA�AQUE,METRO MANILA,high,95
9996,Structured_Philippine_Addresses_Unmatched_part...,bry. graceville sjdm,BARANGAY. GRACEVILLE SAN JOSE DEL MONTE CITY,CITY OF SAN JOSE DEL MONTE,BULACAN,high,95
9997,Structured_Philippine_Addresses_Unmatched_part...,"Westwood 5, Block 11, Lot 9, Nightjar Street","WESTWOOD 5, BLOCK 11, LOT 9, NIGHTJAR STREET",CITY OF ILOILO,ILOILO,high,95
9998,Structured_Philippine_Addresses_Unmatched_part...,BLK10 LOT07 Woodview,BLK10 LOT07 WOODVIEW,WAO,LANAO DEL SUR,medium,72


In [39]:
# Snapshot fuzzy outputs before API mutates records
pre_api_df = pd.DataFrame([dict(r) for r in records])
print(f'Captured pre_api_df rows: {len(pre_api_df):,}')

Captured pre_api_df rows: 10,000


## Cell 8 -- API stage disabled

API verification is intentionally disabled in this fast run profile.

This notebook now runs in fuzzy-only mode using:
- `dim_location` for PSGC hierarchy mapping
- `ph_address_alias_extended_v4.csv` for normalization

If API is needed later, re-enable it in configuration and restore the API stage logic.


In [40]:
# API stage intentionally disabled for maximum throughput in fuzzy-only mode.
# Keep this cell as a no-op so downstream cells continue to run without changes.

USE_API = False
log.info('API stage skipped (fuzzy-only mode).')

for row in low_conf:
    row['api_status'] = 'skipped'
    row['needs_api'] = False

log.info('Proceeding directly to merge/output stages.')


15:45:40  INFO      API stage skipped (fuzzy-only mode).
15:45:40  INFO      Proceeding directly to merge/output stages.


## Cell 9 — Stage 5a: Merge results & determine validity

Combines the processed rows and applies strict validity rules.

- **When `USE_API=True`**: valid only if API verification resolved city/province (`api_status` in `matched`, `province_only`)
- **When `USE_API=False`**: fallback fuzzy-only validity logic is used

This ensures rows with unresolved location after API retries are marked invalid.

In [41]:
all_records = low_conf if USE_API else (high_conf + low_conf)
out_df = pd.DataFrame(all_records)

# Final validity flag
# - API mode: valid only when API verification produced city/province mapping
# - Fuzzy mode: keep previous rule
if USE_API:
    out_df['is_valid'] = (
        out_df['api_status'].isin(['matched', 'province_only']) &
        (
            out_df['city_name'].notna() |
            out_df['province_name'].notna()
        )
    )
else:
    out_df['is_valid'] = (
        out_df['city_name'].notna() |
        (
            out_df['province_name'].notna() &
            (out_df['province_score'] >= PROV_SCORE_HIGH)
        )
    )

# Confidence value counts
print('Confidence distribution:')
print(out_df['confidence'].value_counts().to_string())
if 'api_status' in out_df.columns:
    print('\nAPI status distribution:')
    print(out_df['api_status'].value_counts().to_string())
print(f'\nValid   : {out_df["is_valid"].sum():,}')
print(f'Invalid : {(~out_df["is_valid"]).sum():,}')
print(f'Total   : {len(out_df):,}')


Confidence distribution:
confidence
high      5673
medium    3462
low        865

API status distribution:
api_status
skipped    10000

Valid   : 9,135
Invalid : 865
Total   : 10,000


In [42]:
# City-only sanity check: barangay should remain blank when not explicit
city_only_mask = out_df['original_address'].astype(str).str.lower().isin(['vigan ilocos sur', 'loilo city'])
display(out_df.loc[city_only_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code


In [43]:
# Detailed-address check: barangay should still be populated when appropriate
detail_mask = out_df['original_address'].astype(str).str.contains(
    'Batasan|South Signal|Kim View|sapang palay', case=False, na=False
)
display(out_df.loc[detail_mask, [
    'original_address', 'city_name', 'province_name', 'barangay_name', 'barangay_code'
]])

,original_address,city_name,province_name,barangay_name,barangay_code
338,8 MANGAHAN ST ZONE South Signal,SOUTH UBIAN,TAWI-TAWI,NaN,NaN
529,AJ BATASAN MATANDA,SAN MIGUEL,BULACAN,BATASAN MATANDA,9.0
1247,South signal Village,CITY OF TAGUIG,METRO MANILA,SOUTH SIGNAL VILLAGE,28.0
2267,Sapang Palay,CITY OF SAN JOSE DEL MONTE,BULACAN,SAPANG PALAY,10.0
5288,sapang palay proper,SAPANG DALAGA,MISAMIS OCCIDENTAL,NaN,NaN
6590,40a rodman st batasan,BALASAN,ILOILO,NaN,NaN
7235,10 Mt Elgon St Filinvest 1 Batasan,BALASAN,ILOILO,NaN,NaN
8239,3 PINAGLABANAN SREET BRGY BATASAN HILLS CORNER...,BALASAN,ILOILO,NaN,NaN
8965,199 Dmayan St Batasan,BALASAN,ILOILO,NaN,NaN
9149,51 kasayahan st batasan hills,BALASAN,ILOILO,NaN,NaN


In [44]:
# Before-vs-after change report (fuzzy baseline vs API-final)
compare_cols = ['city_name', 'province_name', 'confidence', 'api_status']
base = pre_api_df[['original_address'] + compare_cols].copy()
base = base.rename(columns={
    'city_name': 'city_before',
    'province_name': 'province_before',
    'confidence': 'confidence_before',
    'api_status': 'api_status_before',
})

final = out_df[['original_address'] + compare_cols].copy()
final = final.rename(columns={
    'city_name': 'city_after',
    'province_name': 'province_after',
    'confidence': 'confidence_after',
    'api_status': 'api_status_after',
})

change_df = base.merge(final, on='original_address', how='inner')
changed_mask = (
    change_df['city_before'].fillna('') != change_df['city_after'].fillna('')
) | (
    change_df['province_before'].fillna('') != change_df['province_after'].fillna('')
) | (
    change_df['confidence_before'].fillna('') != change_df['confidence_after'].fillna('')
) | (
    change_df['api_status_before'].fillna('') != change_df['api_status_after'].fillna('')
)

changed_rows = change_df[changed_mask].reset_index(drop=True)
print(f'Rows changed after API verification: {len(changed_rows):,} / {len(change_df):,}')
display(changed_rows)

Rows changed after API verification: 0 / 10,000


,original_address,city_before,province_before,confidence_before,api_status_before,city_after,province_after,confidence_after,api_status_after


In [45]:
# Compact delta view
compact = changed_rows[[
    'original_address',
    'city_before', 'city_after',
    'province_before', 'province_after',
    'api_status_before', 'api_status_after'
]].copy()
compact['original_address'] = compact['original_address'].astype(str).str.slice(0, 70)
display(compact)

# Highlight the user's example row
mask = changed_rows['original_address'].astype(str).str.contains('General Espino South Signal', case=False, na=False)
if mask.any():
    print('General Espino South Signal change:')
    display(changed_rows.loc[mask, [
        'original_address',
        'city_before', 'city_after',
        'province_before', 'province_after',
        'confidence_before', 'confidence_after',
        'api_status_before', 'api_status_after'
    ]])

,original_address,city_before,city_after,province_before,province_after,api_status_before,api_status_after


## Cell 10 — Stage 5b: Export to Excel

Creates three output files:
- `output/verified/verified_addresses.xlsx`
- `output/invalid/invalid_addresses.xlsx`
- `output/combined_addresses.xlsx`

Rules:
- Verified/Invalid files use only the base address hierarchy columns in the required order.
- Combined file includes `Normalized_Address` plus base columns, PSGC/confidence/scoring/OSM diagnostics, and a `status` column.

In [46]:
from pathlib import Path

# ── Output targets ─────────────────────────────────────────────────────────────
out_root = Path(BASE_DIR) / 'output'
out_verified_dir = out_root / 'verified'
out_invalid_dir = out_root / 'invalid'
out_verified_dir.mkdir(parents=True, exist_ok=True)
out_invalid_dir.mkdir(parents=True, exist_ok=True)

# Output suffix based on batch parts (e.g., parts_001_010)
batch_nums = []
for p in input_paths:
    m = re.search(r'part_(\d+)', p.name.lower())
    if m:
        batch_nums.append(int(m.group(1)))

if batch_nums:
    batch_suffix = f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
elif input_paths:
    batch_suffix = f'batch_{len(input_paths):03d}'
else:
    batch_suffix = 'single'

VERIFIED_FILE = str(out_verified_dir / f'verified_addresses_{batch_suffix}.xlsx')
INVALID_FILE = str(out_invalid_dir / f'invalid_addresses_{batch_suffix}.xlsx')
COMBINED_FILE = str(out_root / f'combined_addresses_{batch_suffix}.xlsx')

# ── Column order required by business template (image) ───────────────────────
BASE_EXPORT_COLS = [
    'Original_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
]

# Build standardized base table from pipeline output
base_df = out_df.copy()
base_df = base_df.rename(columns={
    'original_address': 'Original_Address',
    'barangay_name': 'barangay',
    'city_name': 'city',
    'province_name': 'province',
})

# Ensure expected columns exist even when null in invalid rows
for c in BASE_EXPORT_COLS:
    if c not in base_df.columns:
        base_df[c] = None

verified_df = base_df[base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)
invalid_df = base_df[~base_df['is_valid']][BASE_EXPORT_COLS].reset_index(drop=True)

# Combined file keeps additional diagnostics + status
combined_cols = [
    'Original_Address',
    'Normalized_Address',
    'barangay_code', 'barangay',
    'city_code', 'city',
    'province_code', 'province',
    'region_code', 'region_name',
    'psgc_10',
    'confidence',
    'city_score',
    'province_score',
    'osm_display',
    'osm_lat',
    'osm_lon',
    'status',
]

combined_df = base_df.copy()
combined_df['Normalized_Address'] = combined_df.get('address_normalized')
combined_df['psgc_10'] = combined_df.get('psgc_10_digit')
combined_df['status'] = np.where(combined_df['is_valid'], 'verified', 'invalid')
for c in combined_cols:
    if c not in combined_df.columns:
        combined_df[c] = None
combined_df = combined_df[combined_cols].reset_index(drop=True)


def write_xlsx(df: pd.DataFrame, path: str, sheet_name: str,
               tab_color: str, font_color: str = '#000000'):
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        wb = writer.book
        ws = writer.sheets[sheet_name]
        ws.set_tab_color(tab_color)
        hdr_fmt = wb.add_format({
            'bold': True, 'bg_color': tab_color,
            'font_color': font_color,
            'border': 1, 'text_wrap': True,
            'font_name': 'Arial', 'font_size': 10,
        })
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, hdr_fmt)
            if df.empty:
                max_w = len(col) + 4
            else:
                col_max = df[col].dropna().astype(str).str.len().max()
                if pd.isna(col_max):
                    col_max = len(col)
                max_w = max(int(col_max), len(col)) + 4
            ws.set_column(i, i, min(int(max_w), 42))
        ws.freeze_panes(1, 0)
    log.info(f'Wrote {len(df):,} rows -> {path}')


write_xlsx(verified_df, VERIFIED_FILE, 'Verified', '#00B050')
write_xlsx(invalid_df, INVALID_FILE, 'Invalid', '#FF0000', font_color='#FFFFFF')
write_xlsx(combined_df, COMBINED_FILE, 'Combined', '#1F4E78', font_color='#FFFFFF')

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed:.1f}s  ({elapsed/60:.2f} min)')
print('Output files:')
print(f'  ✅  {VERIFIED_FILE}  ({len(verified_df):,} rows)')
print(f'  ⚠️   {INVALID_FILE}   ({len(invalid_df):,} rows)')
print(f'  📄  {COMBINED_FILE}  ({len(combined_df):,} rows)')


15:45:43  INFO      Wrote 9,135 rows -> ..\..\data\output\verified\verified_addresses_parts_001_010.xlsx
15:45:43  INFO      Wrote 865 rows -> ..\..\data\output\invalid\invalid_addresses_parts_001_010.xlsx
15:45:46  INFO      Wrote 10,000 rows -> ..\..\data\output\combined_addresses_parts_001_010.xlsx



Total elapsed: 1516.4s  (25.27 min)
Output files:
  ✅  ..\..\data\output\verified\verified_addresses_parts_001_010.xlsx  (9,135 rows)
  ⚠️   ..\..\data\output\invalid\invalid_addresses_parts_001_010.xlsx   (865 rows)
  📄  ..\..\data\output\combined_addresses_parts_001_010.xlsx  (10,000 rows)


## Cell 11 — Results summary

Quick inline preview of both output tables for a final sanity check.

In [47]:
print('═' * 80)
print('  PIPELINE SUMMARY')
print('═' * 80)
total = len(out_df)
n_v   = len(verified_df)
n_i   = len(invalid_df)
print(f'  Input rows        : {total:>8,}')
print(f'  Verified          : {n_v:>8,}  ({100*n_v/total:.1f}%)')
print(f'  Invalid           : {n_i:>8,}  ({100*n_i/total:.1f}%)')
print()
print('  Confidence breakdown:')
for conf, cnt in out_df['confidence'].value_counts().items():
    print(f'    {conf:<20} {cnt:>6,}  ({100*cnt/total:.1f}%)')
print()
high_c = out_df[out_df['confidence'] == 'high']
if len(high_c):
    print(f'  Avg city score (high-conf): {high_c["city_score"].mean():.1f}')
print(f'  Total elapsed     : {elapsed:.1f}s')
print('═' * 80)
print()
print('── Verified (first 10 rows) ──')
display(verified_df.head(10))
print()
print('── Invalid (all rows) ──')
display(invalid_df)


════════════════════════════════════════════════════════════════════════════════
  PIPELINE SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Input rows        :   10,000
  Verified          :    9,135  (91.3%)
  Invalid           :      865  (8.7%)

  Confidence breakdown:
    high                  5,673  (56.7%)
    medium                3,462  (34.6%)
    low                     865  (8.7%)

  Avg city score (high-conf): 93.5
  Total elapsed     : 1516.4s
════════════════════════════════════════════════════════════════════════════════

── Verified (first 10 rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,LIONS PARK RES SUNVALLEY PQUE,15.0,SUN VALLEY,4.0,CITY OF PARA�AQUE,76.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
1,"Villa Del Rio Babag 2, Blk 10, Lot 35",13.0,BABAG,2.0,CITY OF BUTUAN,2.0,AGUSAN DEL NORTE,16.0,REGION XIII (CARAGA)
2,zone 1 upper balulng,28.0,ZONE 1,3.0,CITY OF DIGOS,24.0,DAVAO DEL SUR,11.0,REGION XI (DAVAO REGION)
3,2248 Angel Linao St,9.0,LINAO,21.0,TANGCAL,35.0,LANAO DEL NORTE,10.0,REGION X (NORTHERN MINDANAO)
4,1206 Don Quiote St,18.0,BURGOS STREET,27.0,MANGATAREM,55.0,PANGASINAN,1.0,REGION I (ILOCOS REGION)
5,424 D NAVARRO ST TONDO MANILA,NaN,NaN,1.0,TONDO I/II,39.0,METRO MANILA,13.0,NATIONAL CAPITAL REGION (NCR)
6,The columns Ayala Avenue,NaN,NaN,8.0,NORTHERN KABUNTALAN,87.0,MAGUINDANAO DEL NORTE,19.0,BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANA...
7,"MOLO BOLIVARD ILOI,LO",NaN,NaN,12.0,POLOMOLOK,63.0,SOUTH COTABATO,12.0,REGION XII (SOCCSKSARGEN)
8,Baldwin st blk 61 lot 15 a and b timog reside...,2.0,ARELLANO STREET,27.0,MANGATAREM,55.0,PANGASINAN,1.0,REGION I (ILOCOS REGION)
9,19 Twin Hills St New Manila,NaN,NaN,10.0,NEW BATAAN,82.0,DAVAO DE ORO,11.0,REGION XI (DAVAO REGION)



── Invalid (all rows) ──


,Original_Address,barangay_code,barangay,city_code,city,province_code,province,region_code,region_name
0,L25 GREEN MEADOWS SUBD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,B1 l4 pamville subd banaba smr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J2 Jacinto St Jacobs Apartment,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Brgy Road Bibincahan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Phase 4 Citihomes Subd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
860,1009 MATIMYAS ST SAMPALOC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
861,2-6 1st avenue plaridel2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
862,suntrust parkview,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
863,Craig St Sampaloc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cell 12 — One-shot re-run helper

Use this cell to re-run the entire pipeline in one shot after changing config in Cell 1.  
It re-uses all the functions defined above — no need to re-run Cells 3–6 unless you change the reference files.

In [48]:
def run_pipeline(
    input_file   = INPUT_FILE,
    dim_path     = DIM_LOCATION,
    alias_path   = ALIAS_FILE,
    out_verified = OUT_VERIFIED,
    out_invalid  = OUT_INVALID,
    max_rows     = MAX_ROWS,
    use_api      = False,
    api_query_mode = API_QUERY_MODE,
):
    t0 = time.time()
    _dim, _alias, _cities, _provinces, _regions, _barangays = load_reference(
        dim_path, alias_path
    )
    _re, _rmap = build_alias_regex(_alias)

    df = pd.read_excel(input_file)
    if max_rows:
        df = df.iloc[:max_rows]
    col = df.columns[0]

    addresses = df[col].fillna('').astype(str).tolist()
    normalized = [normalize_alias(x, _re, _rmap) for x in addresses]

    recs = []
    for i, addr_norm in enumerate(tqdm(normalized, total=len(normalized), desc='Fuzzy', unit='row')):
        rec = {
            'original_address': addresses[i],
            'address_normalized': addr_norm,
            'api_status': 'skipped',
            'osm_city': None, 'osm_province': None, 'osm_region': None,
            'osm_display': None, 'osm_lat': None, 'osm_lon': None,
        }
        rec.update(match_address(
            addr_norm,
            _dim, _cities, _provinces, _regions, _barangays,
        ))
        rec['needs_api'] = False
        recs.append(rec)

    if use_api:
        log.warning('API path is disabled in this fast profile; running fuzzy-only instead')

    merged = pd.DataFrame(recs)

    merged['is_valid'] = (
        merged['city_name'].notna() |
        (merged['province_name'].notna() & (merged['province_score'] >= PROV_SCORE_HIGH))
    )

    v_df = merged[merged['is_valid']][
        [c for c in VERIFIED_COLS if c in merged.columns]
    ].reset_index(drop=True)
    i_df = merged[~merged['is_valid']][
        [c for c in INVALID_COLS if c in merged.columns]
    ].reset_index(drop=True)

    write_xlsx(v_df, out_verified, 'Verified', '#00B050')
    write_xlsx(i_df, out_invalid, 'Invalid', '#FF0000', font_color='#FFFFFF')

    elapsed = time.time() - t0
    log.info(f'Done in {elapsed:.1f}s  |  Verified: {len(v_df):,}  Invalid: {len(i_df):,}')
    return v_df, i_df


# Uncomment to run:
# verified_df, invalid_df = run_pipeline()
# verified_df, invalid_df = run_pipeline(max_rows=1000)
